In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ParquetTest") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

try:
    df = spark.read.parquet("D:/data engineer/pyspark_practice/pyspark_scenarios/dataset1_transactions/transactions")
    df.show()
except Exception as e:
    print(e)

#df = spark.read.option("mergeSchema", "true").parquet("D:/data engineer/pyspark_practice/pyspark_scenarios/dataset1_transactions/transactions")

#df.show(5)


spark.stop()

+--------------+-----------+-----------+--------+---------+-------+--------------+-------------------+
|transaction_id|customer_id|merchant_id|  amount|   status| region|payment_method|     transaction_ts|
+--------------+-----------+-----------+--------+---------+-------+--------------+-------------------+
|   T0000908385|   C0244020|     M00010| 9851.15|completed|   West|   Net Banking|2024-01-01 00:00:14|
|   T0000656247|   C0207485|     M00114|12940.25|   failed|   East|    Debit Card|2024-01-01 00:00:16|
|   T0000989639|   C0218336|     M00145|12356.47|  pending|  North|    Debit Card|2024-01-01 00:00:16|
|   T0000344253|   C0182286|     M00010|10590.22|completed|   East|   Credit Card|2024-01-01 00:00:16|
|   T0000348331|   C0220739|     M00164|18015.78|   failed|  North|   Net Banking|2024-01-01 00:00:20|
|   T0000539983|   C0190822|     M00424| 5230.45| refunded|  North|   Credit Card|2024-01-01 00:00:21|
|   T0000471264|   C0196913|     M00001| 8533.28|completed|Central|   Cre

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import*
spark = SparkSession.builder \
    .appName("ParquetTest") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
df = spark.read.parquet("D:/data engineer/pyspark_practice/pyspark_scenarios/dataset1_transactions/merchant_lookup/merchant_lookup.parquet")
df.show()
df.filter(col("tier")=="gold").select(col("merchant_name"),col("category"))

+-----------+---------------+-------------+------+
|merchant_id|  merchant_name|     category|  tier|
+-----------+---------------+-------------+------+
|     M00001|Merchant_M00001|       Sports|Bronze|
|     M00002|Merchant_M00002|       Travel|Silver|
|     M00003|Merchant_M00003|  Electronics|Bronze|
|     M00004|Merchant_M00004|  Electronics|Silver|
|     M00005|Merchant_M00005|        Books|Silver|
|     M00006|Merchant_M00006|Food & Dining|Silver|
|     M00007|Merchant_M00007|Entertainment|Silver|
|     M00008|Merchant_M00008|     Clothing|Silver|
|     M00009|Merchant_M00009|Food & Dining|  Gold|
|     M00010|Merchant_M00010|Food & Dining|  Gold|
|     M00011|Merchant_M00011|  Electronics|Bronze|
|     M00012|Merchant_M00012|    Groceries|Bronze|
|     M00013|Merchant_M00013|       Sports|  Gold|
|     M00014|Merchant_M00014|  Electronics|  Gold|
|     M00015|Merchant_M00015|Entertainment|Bronze|
|     M00016|Merchant_M00016|Entertainment|Bronze|
|     M00017|Merchant_M00017|  

DataFrame[merchant_name: string, category: string]

In [ ]:
#1
skew_df = df.groupBy("merchant_id") \
    .agg(count("*").alias("cnt"))
skew_df.cache()
#2
hot_keys = skew_df.filter(col("cnt") > 100000) \
    .select("merchant_id")
hot_keys.show()


df1 = df.join(broadcast(hot_keys), "merchant_id", "left_semi")
df2 = df.join(broadcast(hot_keys), "merchant_id", "left_anti")

salt_df = df1.withColumn("salt_key",concat(col("merchant_id"),lit("_"),(rand()*100).cast('int').cast('string')))


salt_group =salt_df.groupBy("salt_key").agg(sum("amount").alias("salt_amount"))

#4
total_salted = salt_group.withColumn("merchant_id",split(col("salt_key"),"_").getItem(0)).groupBy("merchant_id").agg(sum("salt_amount").alias("total amount"))
total_without_salted = df2.groupBy("merchant_id").agg(sum("amount").alias("total amount"))

#5
df_result = total_salted.unionByName(total_without_salted).show()

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").config("spark.hadoop.io.native.lib.available", "false").getOrCreate()
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# ── Create Spark Session ─────────────────────────────
spark = SparkSession.builder \
    .appName("Fraud detaction") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# ── Read Data ───────────────────────────────────────
df = spark.read.option("mergeSchema", "true").parquet(
    r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions"
)
df.printSchema()
df.show()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- transaction_ts: timestamp_ntz (nullable = true)
 |-- transaction_amount: double (nullable = true)

+--------------+-----------+-----------+--------+---------+-------+--------------+-------------------+------------------+
|transaction_id|customer_id|merchant_id|  amount|   status| region|payment_method|     transaction_ts|transaction_amount|
+--------------+-----------+-----------+--------+---------+-------+--------------+-------------------+------------------+
|   T0000908385|   C0244020|     M00010| 9851.15|completed|   West|   Net Banking|2024-01-01 00:00:14|              NULL|
|   T0000656247|   C0207485|     M00114|12940.25|   failed|   East|    Debit Card|2024-01-01 00:00:16|         